# Лабораторна робота 4

In [1]:
# %% [Р1] Імпорт, завантаження, вимоги до якості (пороги як конфігурація)
import sys, re, warnings
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
warnings.filterwarnings("ignore"); matplotlib.use("Agg")
RANDOM_STATE = 42
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
raw = pd.read_excel("../data/online_retail.xlsx", dtype={"InvoiceNo": str, "StockCode": str})
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| shape:", raw.shape)
# Вимоги: вимір -> (метрика, поріг); дані для RFM-сегментації клієнтів і звітів продажів
REQ = pd.DataFrame([
    ["Повнота", "CustomerID", "1 - n_null/n", 0.95],
    ["Повнота", "Description", "1 - n_null/n", 0.99],
    ["Унікальність", "запис (усі поля)", "1 - n_dup/n", 0.999],
    ["Валідність", "Quantity", "частка Quantity>0 серед продажів", 0.99],
    ["Валідність", "UnitPrice", "частка UnitPrice>0", 0.99],
    ["Валідність", "StockCode", "частка ^\\d{5}[A-Za-z]{0,2}$", 0.98],
    ["Узгодженість", "InvoiceNo~Quantity", "частка (C ⇔ Q<0)", 0.99],
    ["Актуальність", "InvoiceDate", "середнє max(0,1-age/365)", 0.50]],
    columns=["вимір", "поле", "метрика", "поріг"])
print(REQ.to_string(index=False))

Python 3.12.16 | pandas 3.1.1 | shape: (541909, 8)
       вимір               поле                          метрика  поріг
     Повнота         CustomerID                     1 - n_null/n  0.950
     Повнота        Description                     1 - n_null/n  0.990
Унікальність   запис (усі поля)                      1 - n_dup/n  0.999
  Валідність           Quantity частка Quantity>0 серед продажів  0.990
  Валідність          UnitPrice               частка UnitPrice>0  0.990
  Валідність          StockCode      частка ^\d{5}[A-Za-z]{0,2}$  0.980
Узгодженість InvoiceNo~Quantity                 частка (C ⇔ Q<0)  0.990
Актуальність        InvoiceDate         середнє max(0,1-age/365)  0.500


In [2]:
# %% [Р2.1] Функція profile(df)
def profile(df):
    rows = []
    for c in df.columns:
        s = df[c]
        num = pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s)
        rows.append({"стовпець": c, "тип": str(s.dtype),
                     "пропуски,%": round(s.isna().mean() * 100, 2),
                     "унікальних": s.nunique(),
                     "унікальність,%": round(s.nunique() / len(s) * 100, 2),
                     "min": s.min() if num else "", "max": s.max() if num else "",
                     "top-5": ", ".join(f"{k}({v})" for k, v in
                                        s.value_counts().head(5).items())[:70]})
    return pd.DataFrame(rows)
prof = profile(raw)
print(prof.to_string(index=False))

   стовпець            тип  пропуски,%  унікальних  унікальність,%                 min                 max                                                                  top-5
  InvoiceNo            str        0.00       25900            4.78                                               573585(1114), 581219(749), 581492(731), 580729(721), 558475(705)
  StockCode            str        0.00        4070            0.75                                              85123A(2313), 22423(2203), 85099B(2159), 47566(1727), 20725(1639)
Description         object        0.27        4223            0.78                                         WHITE HANGING HEART T-LIGHT HOLDER(2369), REGENCY CAKESTAND 3 TIER(220
   Quantity          int64        0.00         722            0.13              -80995               80995                     1(148227), 2(81829), 12(61063), 6(40868), 4(38484)
InvoiceDate datetime64[us]        0.00       23260            4.29 2010-12-01 08:26:00 2011-12-09 12:50:00 201

In [3]:
# %% [Р2.2] Приховані пропуски (заглушки) -> NaN
df = raw.copy()
desc = df.Description.astype("string").str.strip()
# у довіднику всі назви товарів — ВЕЛИКИМИ літерами; рядки лише з малих літер ('check',
# 'damaged', 'wet?') чи зі знаків '?' — службові нотатки складу, тобто приховані пропуски
stub_desc = (desc.str.match(r"^\?+$") | (desc == desc.str.lower())).fillna(False).astype(bool)
stubs = {"UnitPrice == 0": (df.UnitPrice == 0).sum(),
         "Description-заглушки ('?', нотатки малими літерами)": stub_desc.sum(),
         "Country == 'Unspecified'": (df.Country == "Unspecified").sum(),
         "CustomerID порожній": df.CustomerID.isna().sum()}
print(pd.Series(stubs, name="к-сть").to_string())
print("Приклади заглушок Description:", df.loc[stub_desc, "Description"].value_counts().head(8).to_dict())
df.loc[df.UnitPrice == 0, "UnitPrice"] = np.nan
df.loc[stub_desc, "Description"] = np.nan
df.loc[df.Country == "Unspecified", "Country"] = np.nan
print("\nЧастка пропусків після заміни заглушок, %:")
print((df.isna().mean() * 100).round(2).to_string())

UnitPrice == 0                                           2515
Description-заглушки ('?', нотатки малими літерами)       549
Country == 'Unspecified'                                  446
CustomerID порожній                                    135080
Приклади заглушок Description: {'check': 159, '?': 47, 'damages': 45, 'damaged': 43, 'found': 25, 'sold as set on dotcom': 20, 'adjustment': 16, 'thrown away': 9}

Частка пропусків після заміни заглушок, %:
InvoiceNo       0.00
StockCode       0.00
Description     0.37
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.46
CustomerID     24.93
Country         0.08


In [4]:
# %% [Р2.3] Карта пропусків
smp = df.sort_values("InvoiceDate").iloc[::200]
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [2, 1]})
sns.heatmap(smp.isna().T, cbar=False, cmap=["#EEEEEE", "#C44E52"], ax=axs[0])
axs[0].set_xticks([]); axs[0].set_xlabel("записи (кожен 200-й, за часом)")
axs[0].set_title("Карта пропусків: CustomerID — основне проблемне поле")
(df.isna().mean() * 100).sort_values().plot.barh(ax=axs[1], color="#C44E52")
axs[1].set_xlabel("% пропусків"); axs[1].set_title("Частка пропусків за стовпцями")
fig.tight_layout(); fig.savefig(FIG / "l4_missing.png", dpi=150)
row_na = df.isna().sum(axis=1)
print("Записів з ≥1 пропуском:", (row_na > 0).sum(), f"({(row_na > 0).mean():.1%})",
      "| з ≥2:", (row_na >= 2).sum())

Записів з ≥1 пропуском: 135364 (25.0%) | з ≥2: 2677


In [5]:
# %% [Р3.1] Метрики якості за вимірами
df["is_cancel"] = df.InvoiceNo.str.startswith("C")
REF_DATE = df.InvoiceDate.max(); VOLATILITY = 365
def metrics(d):
    sales = d[~d.is_cancel]
    age = (REF_DATE - d.InvoiceDate).dt.days
    return pd.Series({
        "Повнота CustomerID": d.CustomerID.notna().mean(),
        "Повнота Description": d.Description.notna().mean(),
        "Унікальність записів": 1 - d.duplicated(subset=raw.columns.tolist()).mean(),
        "Валідність Quantity": (sales.Quantity > 0).mean(),
        "Валідність UnitPrice": (d.UnitPrice > 0).mean(),
        "Валідність StockCode": d.StockCode.astype(str).str.match(r"^\d{5}[A-Za-z]{0,2}$").mean(),
        "Узгодженість C⇔Q<0": (d.is_cancel == (d.Quantity < 0)).mean(),
        "Актуальність (Піпіно)": np.maximum(0, 1 - age / VOLATILITY).mean()})
m_all = metrics(df)
tab = REQ.copy(); tab["значення"] = m_all.values.round(4)
tab["статус"] = np.where(tab.значення >= tab.поріг, "OK", "НЕ ОК")
print(tab.to_string(index=False))
print("\nНевалідні StockCode:", df.loc[~df.StockCode.astype(str).str.match(r"^\d{5}[A-Za-z]{0,2}$"),
                                     "StockCode"].value_counts().head(8).to_dict())

       вимір               поле                          метрика  поріг  значення статус
     Повнота         CustomerID                     1 - n_null/n  0.950    0.7507  НЕ ОК
     Повнота        Description                     1 - n_null/n  0.990    0.9963     OK
Унікальність   запис (усі поля)                      1 - n_dup/n  0.999    0.9903  НЕ ОК
  Валідність           Quantity частка Quantity>0 серед продажів  0.990    0.9975     OK
  Валідність          UnitPrice               частка UnitPrice>0  0.990    0.9954     OK
  Валідність          StockCode      частка ^\d{5}[A-Za-z]{0,2}$  0.980    0.9945     OK
Узгодженість InvoiceNo~Quantity                 частка (C ⇔ Q<0)  0.990    0.9975     OK
Актуальність        InvoiceDate         середнє max(0,1-age/365)  0.500    0.5691     OK

Невалідні StockCode: {'POST': 1256, 'DOT': 710, 'M': 571, 'C2': 144, 'D': 77, 'S': 63, 'BANK CHARGES': 37, 'AMAZONFEE': 34}


In [6]:
# %% [Р3.2] Метрики у зрізах: за країною та за місяцем
df["month"] = df.InvoiceDate.dt.to_period("M").astype(str)
top_c = df.Country.value_counts().head(8).index
def by_slice(d, col):   # метрики для кожного значення зрізу
    return pd.DataFrame({k: metrics(g) for k, g in d.groupby(col)}).T
by_country = by_slice(df[df.Country.isin(top_c)], "Country")
by_month = by_slice(df, "month")
show = ["Повнота CustomerID", "Унікальність записів", "Валідність UnitPrice", "Узгодженість C⇔Q<0"]
print(by_country[show].round(3).to_string())
print(); print(by_month[show].round(3).to_string())
fig, axs = plt.subplots(1, 2, figsize=(14, 4.2))
by_month["Повнота CustomerID"].plot(ax=axs[0], marker="o", label="за місяцями")
axs[0].axhline(m_all["Повнота CustomerID"], color="k", ls="--", label="середнє")
axs[0].axhline(0.95, color="r", ls=":", label="поріг 0.95")
axs[0].set_ylim(0, 1.02); axs[0].set_ylabel("повнота CustomerID"); axs[0].legend()
axs[0].set_title("Провал повноти у грудні 2010 – січні 2011 (≈ 62 %)")
axs[0].tick_params(axis="x", rotation=45)
by_country["Повнота CustomerID"].sort_values().plot.barh(ax=axs[1], color="#4C72B0")
axs[1].axvline(0.95, color="r", ls=":"); axs[1].set_xlim(0, 1.02)
axs[1].set_xlabel("повнота CustomerID"); axs[1].set_title("UK (73 %) — головне джерело пропусків CustomerID")
fig.tight_layout(); fig.savefig(FIG / "l4_slices.png", dpi=150)

                Повнота CustomerID  Унікальність записів  Валідність UnitPrice  Узгодженість C⇔Q<0
Belgium                      1.000                 1.000                 1.000               1.000
EIRE                         0.913                 0.999                 1.000               1.000
France                       0.992                 0.998                 1.000               1.000
Germany                      1.000                 0.998                 1.000               1.000
Netherlands                  1.000                 1.000                 0.998               1.000
Spain                        1.000                 0.998                 1.000               1.000
Switzerland                  0.938                 0.996                 1.000               1.000
United Kingdom               0.730                 0.990                 0.995               0.997

         Повнота CustomerID  Унікальність записів  Валідність UnitPrice  Узгодженість C⇔Q<0
2010-12         

In [7]:
# %% [Р4.1] Автоматизовані перевірки: 12 правил на трьох рівнях
CODE_RE = r"^\d{5}[A-Za-z]{0,2}$"
RULES = [  # (id, рівень, опис, критичність, функція -> маска порушень або bool)
    ("F1", "поле", "InvoiceNo: 6 цифр, можливо з префіксом C/A", "критичне",
     lambda d: ~d.InvoiceNo.str.match(r"^[CA]?\d{6}$")),
    ("F2", "поле", "StockCode відповідає формату товарного коду", "некритичне",
     lambda d: ~d.StockCode.astype(str).str.match(CODE_RE)),
    ("F3", "поле", "UnitPrice > 0 і < 10000", "критичне",
     lambda d: ~d.UnitPrice.between(0.001, 10000)),
    ("F4", "поле", "|Quantity| < 10000", "критичне", lambda d: d.Quantity.abs() >= 10000),
    ("F5", "поле", "CustomerID не порожній", "некритичне", lambda d: d.CustomerID.isna()),
    ("R1", "запис", "скасування (C) ⇔ Quantity < 0", "критичне",
     lambda d: d.InvoiceNo.str.startswith("C") != (d.Quantity < 0)),
    ("R2", "запис", "продаж з Quantity>0 має Description", "некритичне",
     lambda d: (d.Quantity > 0) & d.Description.isna()),
    ("R3", "запис", "InvoiceDate у межах періоду вивантаження", "критичне",
     lambda d: ~d.InvoiceDate.between("2010-12-01", "2011-12-10")),
    ("S1", "набір", "немає повних дублікатів записів", "критичне",
     lambda d: d.duplicated(subset=raw.columns.tolist())),
    ("S2", "набір", "одна накладна — один клієнт і одна країна", "критичне",
     lambda d: d.InvoiceNo.map(d.groupby("InvoiceNo").Country.nunique()) > 1),
    ("S3", "набір", "обсяг набору ≥ 500 000 записів", "критичне",
     lambda d: pd.Series([len(d) < 500_000])),
    ("S4", "набір", "частка UK у межах 85–95 % (розподіл)", "некритичне",
     lambda d: pd.Series([not 0.85 <= (d.Country == "United Kingdom").mean() <= 0.95])),
]
def run_checks(d):
    rep = []
    for rid, lvl, desc, crit, f in RULES:
        mask = f(d).fillna(False).astype(bool)
        n = int(mask.sum()); share = n / len(mask)
        ex = d.loc[mask.values, ["InvoiceNo", "StockCode", "Quantity", "UnitPrice"]].head(2) \
            .apply(lambda r: "|".join(map(str, r)), axis=1).tolist() if len(mask) == len(d) and n else []
        rep.append([rid, lvl, desc, crit, n, round(share, 4), "; ".join(ex)[:55]])
    return pd.DataFrame(rep, columns=["id", "рівень", "правило", "критичність",
                                      "порушень", "частка", "приклади"])
report = run_checks(df)
print(report.to_string(index=False))

id рівень                                     правило критичність  порушень  частка                                           приклади
F1   поле  InvoiceNo: 6 цифр, можливо з префіксом C/A    критичне         0  0.0000                                                   
F2   поле StockCode відповідає формату товарного коду  некритичне      2995  0.0055              536370|POST|3|18.0; C536379|D|-1|27.5
F3   поле                     UnitPrice > 0 і < 10000    критичне      2527  0.0047            536414|22139|56|nan; 536545|21134|1|nan
F4   поле                          |Quantity| < 10000    критичне         5  0.0000 541431|23166|74215|1.04; C541433|23166|-74215|1.04
F5   поле                      CustomerID не порожній  некритичне    135080  0.2493           536414|22139|56|nan; 536544|21773|1|2.51
R1  запис               скасування (C) ⇔ Quantity < 0    критичне      1336  0.0025        536589|21777|-10|nan; 536764|84952C|-38|nan
R2  запис         продаж з Quantity>0 має Description  

In [8]:
# %% [Р4.2] Поведінка конвеєра: критичні правила зупиняють, некритичні — попереджають
class DataQualityError(Exception): pass
def gate(rep, tolerance=0.0):
    crit = rep[(rep.критичність == "критичне") & (rep.частка > tolerance)]
    warn = rep[(rep.критичність == "некритичне") & (rep.порушень > 0)]
    for _, w in warn.iterrows():
        print(f"WARNING [{w.id}] {w.правило}: {w.порушень} порушень")
    if len(crit):
        raise DataQualityError("STOP: порушено критичні правила " + ", ".join(crit.id))
    print("PASS: конвеєр продовжує роботу")
try:
    gate(report)
except DataQualityError as e:
    print(e)
# після очищення: дублікати, ціни <= 0, екстремальні кількості, неузгоджені скасування
clean = df.drop_duplicates(subset=raw.columns.tolist())
clean = clean[clean.UnitPrice.between(0.001, 10000) & (clean.Quantity.abs() < 10000)]
clean = clean[clean.InvoiceNo.str.startswith("C") == (clean.Quantity < 0)]
print(f"\nПісля очищення: {len(clean)} записів ({len(clean)/len(df):.1%})")
try:
    gate(run_checks(clean))
except DataQualityError as e:
    print(e)

WARNING [F2] StockCode відповідає формату товарного коду: 2995 порушень
WARNING [F5] CustomerID не порожній: 135080 порушень
WARNING [R2] продаж з Quantity>0 має Description: 726 порушень
STOP: порушено критичні правила F3, R1, S1



Після очищення: 534115 записів (98.6%)


WARNING [F2] StockCode відповідає формату товарного коду: 2947 порушень
WARNING [F5] CustomerID не порожній: 132556 порушень
PASS: конвеєр продовжує роботу


In [9]:
# %% [Р5] Матриця якості: вимір — метрика — поріг — факт — статус
m_clean = metrics(clean)
matrix = REQ.copy()
matrix["факт (сирі)"] = m_all.values.round(4); matrix["факт (очищені)"] = m_clean.values.round(4)
matrix["статус"] = np.where(matrix["факт (очищені)"] >= matrix.поріг, "OK", "НЕ ОК")
print(matrix.to_string(index=False))
print("\nУнікальних клієнтів:", clean.CustomerID.nunique(),
      "| частка виручки без CustomerID:",
      round((clean.Quantity * clean.UnitPrice)[clean.CustomerID.isna()].sum()
            / (clean.Quantity * clean.UnitPrice).sum(), 3))

       вимір               поле                          метрика  поріг  факт (сирі)  факт (очищені) статус
     Повнота         CustomerID                     1 - n_null/n  0.950       0.7507          0.7518  НЕ ОК
     Повнота        Description                     1 - n_null/n  0.990       0.9963          1.0000     OK
Унікальність   запис (усі поля)                      1 - n_dup/n  0.999       0.9903          1.0000     OK
  Валідність           Quantity частка Quantity>0 серед продажів  0.990       0.9975          1.0000     OK
  Валідність          UnitPrice               частка UnitPrice>0  0.990       0.9954          1.0000     OK
  Валідність          StockCode      частка ^\d{5}[A-Za-z]{0,2}$  0.980       0.9945          0.9945     OK
Узгодженість InvoiceNo~Quantity                 частка (C ⇔ Q<0)  0.990       0.9975          1.0000     OK
Актуальність        InvoiceDate         середнє max(0,1-age/365)  0.500       0.5691          0.5689     OK

Унікальних клієнтів: 4370 |